# Part 3 — Dự báo Revenue & COGS




In [29]:
%pip install -q prophet lightgbm
import pandas as pd
import numpy as np
import lightgbm as lgb
from prophet import Prophet
from sklearn.linear_model import Ridge
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings, logging
warnings.filterwarnings('ignore')
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [30]:
# === 1. LOAD DATA (Portable — chạy được ở mọi nơi) ===
import os
from pathlib import Path

# Tự động xác định thư mục database
if os.path.exists('/kaggle/input'):
    DB_DIR = Path('/kaggle/input/competitions/datathon-2026-round-1/')
    print('🚀 Đang chạy trên Kaggle')
else:
    DB_DIR = Path.cwd().parent / 'database'
    print('💻 Đang chạy trên máy cá nhân / GitHub Codespaces')

print(f'📂 Thư mục dữ liệu: {DB_DIR}')

sales = pd.read_csv(DB_DIR / 'sales.csv', parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
print(f'Training period: {sales["Date"].min().date()} to {sales["Date"].max().date()}')
sales.head()


💻 Đang chạy trên máy cá nhân / GitHub Codespaces
📂 Thư mục dữ liệu: /Users/tranquoctuan18112005/Documents/DATATHON/datathon-2026-round-1/bốn con cừu/database


Training period: 2012-07-04 to 2022-12-31


,Date,Revenue,COGS
0,2012-07-04,5123547.94,3982991.19
1,2012-07-05,2751773.45,2150580.23
2,2012-07-06,3054029.42,2517632.84
3,2012-07-07,2667930.94,2108246.62
4,2012-07-08,2360851.90,1808622.79


In [31]:
# === 2. FEATURE ENGINEERING ===


PROMO_SCHEDULE = [
    ("spring_sale",    3, 18, 30, 12,   True),
    ("mid_year",       6, 23, 29, 18,   True),
    ("fall_launch",    8, 30, 32, 10,   True),
    ("year_end",      11, 18, 45, 20,   True),
    ("urban_blowout",  7, 30, 33, None, "odd"),
    ("rural_special",  1, 30, 30, 15,   "odd"),
]

TET_DATES = {
    2013: "2013-02-10", 2014: "2014-01-31", 2015: "2015-02-19",
    2016: "2016-02-08", 2017: "2017-01-28", 2018: "2018-02-16",
    2019: "2019-02-05", 2020: "2020-01-25", 2021: "2021-02-12",
    2022: "2022-02-01", 2023: "2023-01-22", 2024: "2024-02-10",
}

VN_FIXED_HOLIDAYS = [
    (1,  1,  "new_year"), (3,  8,  "womens_day"), (4,  30, "reunification"),
    (5,  1,  "labor_day"), (9,  2,  "national_day"), (10, 20, "vn_womens_day"),
    (11, 11, "dd_1111"), (12, 12, "dd_1212"), (12, 24, "christmas_eve"), (12, 25, "christmas"),
]

class FeatureEngineer:
    def __init__(self):
        self._tet_lut = {y: pd.Timestamp(v) for y, v in TET_DATES.items()}

    def build_features(self, dates) -> pd.DataFrame:
        df = pd.DataFrame({"Date": pd.DatetimeIndex(dates)})
        d = df["Date"]
        df["year"]    = d.dt.year
        df["month"]   = d.dt.month
        df["day"]     = d.dt.day
        df["dow"]     = d.dt.dayofweek
        df["doy"]     = d.dt.dayofyear
        df["quarter"] = d.dt.quarter
        df["is_weekend"]    = (df["dow"] >= 5).astype(int)
        df["days_to_eom"]   = d.dt.days_in_month - df["day"]
        df["days_from_som"] = df["day"] - 1
        df["dim"]           = d.dt.days_in_month
        for k in [1, 2, 3]:
            df[f"is_last{k}"]  = (df["days_to_eom"]  <= k - 1).astype(int)
            df[f"is_first{k}"] = (df["days_from_som"] <= k - 1).astype(int)
        df["t_days"]          = (d - pd.Timestamp("2020-01-01")).dt.days
        df["t_years"]         = df["t_days"] / 365.25
        df["regime_pre2019"]  = (df["year"] <= 2018).astype(int)
        df["regime_2019"]     = (df["year"] == 2019).astype(int)
        df["regime_post2019"] = (df["year"] >= 2020).astype(int)
        TAU = 2 * np.pi
        for k in (1, 2, 3, 4, 5):
            df[f"sin_y{k}"] = np.sin(TAU * k * df["doy"] / 365.25)
            df[f"cos_y{k}"] = np.cos(TAU * k * df["doy"] / 365.25)
        for k in (1, 2):
            df[f"sin_w{k}"] = np.sin(TAU * k * df["dow"] / 7.0)
            df[f"cos_w{k}"] = np.cos(TAU * k * df["dow"] / 7.0)
        for k in (1, 2):
            df[f"sin_m{k}"] = np.sin(TAU * k * (df["day"] - 1) / df["dim"])
            df[f"cos_m{k}"] = np.cos(TAU * k * (df["day"] - 1) / df["dim"])
        for (m, dd, name) in VN_FIXED_HOLIDAYS:
            df[f"hol_{name}"] = ((df["month"] == m) & (df["day"] == dd)).astype(int)

        # === TET FEATURES (GIONG NGUYEN + THEM tet_effect) ===
        diffs = np.array([self._nearest_tet_diff(dd) for dd in d])
        df["tet_days_diff"] = diffs
        df["tet_in_7"]      = (np.abs(diffs) <= 7).astype(int)
        df["tet_in_14"]     = (np.abs(diffs) <= 14).astype(int)
        df["tet_before_7"]  = ((diffs >= -7) & (diffs < 0)).astype(int)
        df["tet_after_7"]   = ((diffs > 0) & (diffs <= 7)).astype(int)
        df["tet_on"]        = (diffs == 0).astype(int)
    
        # Giup LightGBM hoc gradient muot hon thay vi binary flags thang dung
        df["tet_effect"]    = np.exp(-np.abs(diffs) / 7.0)

        df["hol_black_friday"] = [self._is_black_friday(dd) for dd in d]
        yrs = sorted(df["year"].unique().tolist())
        for (name, sm, sd, dur, disc, recur) in PROMO_SCHEDULE:
            in_prom = np.zeros(len(df), dtype=int)
            since   = np.full(len(df), -1.0)
            until   = np.full(len(df), -1.0)
            discount= np.zeros(len(df))
            for y in range(min(yrs) - 1, max(yrs) + 2):
                if recur == "odd" and y % 2 == 0: continue
                start = pd.Timestamp(year=y, month=sm, day=sd)
                end   = start + pd.Timedelta(days=dur)
                mask  = (d >= start) & (d <= end)
                in_prom[mask] = 1
                since[mask]   = (d[mask] - start).dt.days
                until[mask]   = (end - d[mask]).dt.days
                discount[mask]= disc or 0
            df[f"promo_{name}"]       = in_prom
            df[f"promo_{name}_since"] = since
            df[f"promo_{name}_until"] = until
            df[f"promo_{name}_disc"]  = discount
        return df

    def build_promo_regressors(self, dates) -> pd.DataFrame:
        full = self.build_features(dates)
        promo_cols = [c for c in full.columns if c.startswith("promo_") and c.count("_") == 1]
        return full[["Date"] + promo_cols].rename(columns={"Date": "ds"})

    def feature_columns(self, feat_df: pd.DataFrame) -> list:
        non_feat = {"Date", "Revenue", "COGS"}
        return [c for c in feat_df.columns if c not in non_feat]

    def _nearest_tet_diff(self, dd):
        cands = [self._tet_lut.get(dd.year), self._tet_lut.get(dd.year - 1), self._tet_lut.get(dd.year + 1)]
        valid = [(dd - c).days for c in cands if c is not None and abs((dd - c).days) <= 45]
        return min(valid) if valid else 999

    @staticmethod
    def _is_black_friday(dd) -> int:
        if dd.month != 11: return 0
        last = pd.Timestamp(year=dd.year, month=11, day=30)
        last_fri = last - pd.Timedelta(days=(last.dayofweek - 4) % 7)
        return int(dd == last_fri)

fe = FeatureEngineer()
print('Feature Engineer ready.')


Feature Engineer ready.


In [ ]:
# === 3. MODELING PIPELINE (GIONG NGUYEN 100% BAN GOC) ===
_LGB_PARAMS = dict(
    objective="regression", metric="mae",
    learning_rate=0.03, num_leaves=63,
    min_data_in_leaf=30,
    feature_fraction=0.85, bagging_fraction=0.85, bagging_freq=5,
    lambda_l2=1.0, seed=42, verbosity=-1,
)

_ALPHA       = 0.60
_CAL_REV     = 1.26  
_CAL_COGS    = 1.32 
_BETA_MARGIN = 0.30
_Q_BOOST     = {1: 2.0, 2: 3.0, 3: 1.5, 4: 2.0}

class ForecastingPipeline:
    def __init__(self, feature_engineer: FeatureEngineer = None):
        self.fe = feature_engineer or FeatureEngineer()
        self._feat_df = None

    def fit(self, sales_df: pd.DataFrame):
        feat = self.fe.build_features(sales_df["Date"])
        feat["Revenue"] = sales_df["Revenue"].values
        feat["COGS"]    = sales_df["COGS"].values
        self._feat_df = feat
        self._feature_cols = self.fe.feature_columns(feat)

        self._X_tr  = feat[self._feature_cols].values.astype(float)
        self._y_rev = np.log(feat["Revenue"].values)  
        self._y_cog = np.log(feat["COGS"].values)    

        years = feat["Date"].dt.year.values
        self._sample_w = np.full(len(years), 0.01)
        self._sample_w[(years >= 2014) & (years <= 2018)] = 1.0  

        print("Training Ridge...")
        self._ridge_rev, self._stats_rev = self._train_ridge(self._X_tr, self._y_rev)
        self._ridge_cog, self._stats_cog = self._train_ridge(self._X_tr, self._y_cog)

        print("Training LightGBM base...")
        self._lgb_rev, _ = self._train_lgb(self._X_tr, self._y_rev, self._sample_w)
        self._lgb_cog, _ = self._train_lgb(self._X_tr, self._y_cog, self._sample_w)

        print("Training Prophet...")
        self._prophet_rev = self._train_prophet(sales_df, target="Revenue")
        self._prophet_cog = self._train_prophet(sales_df, target="COGS")

        print("Training Q-Specialists (8 models)...")
        self._spec_rev, self._spec_cog = self._train_q_specialists()

        print("All models trained.")
        return self

    def predict(self, test_dates) -> pd.DataFrame:
        test_df = self.fe.build_features(test_dates)
        X_te    = test_df[self._feature_cols].values.astype(float)

        p_rd_rev = np.exp(self._apply_ridge(self._ridge_rev, self._stats_rev, X_te)) 
        p_rd_cog = np.exp(self._apply_ridge(self._ridge_cog, self._stats_cog, X_te))
        p_lgb_rev = np.exp(self._lgb_rev.predict(X_te))
        p_lgb_cog = np.exp(self._lgb_cog.predict(X_te))

        vdf = pd.DataFrame({"ds": test_df["Date"]}).merge(
            self.fe.build_promo_regressors(test_df["Date"]), on="ds")
        p_pr_rev = np.exp(self._prophet_rev.predict(vdf)["yhat"].values)
        p_pr_cog = np.exp(self._prophet_cog.predict(vdf)["yhat"].values)

        Q_test = test_df["Date"].dt.quarter.values
        lgb_spec_rev = np.zeros(len(test_dates))
        lgb_spec_cog = np.zeros(len(test_dates))
        for q in [1, 2, 3, 4]:
            mask = Q_test == q
            lgb_spec_rev[mask] = np.exp(self._spec_rev[q].predict(X_te))[mask]
            lgb_spec_cog[mask] = np.exp(self._spec_cog[q].predict(X_te))[mask]

        lgb_blend_rev = _ALPHA * lgb_spec_rev + (1 - _ALPHA) * p_lgb_rev
        lgb_blend_cog = _ALPHA * lgb_spec_cog + (1 - _ALPHA) * p_lgb_cog

        raw_rev = 0.10 * p_pr_rev + 0.10 * p_rd_rev + 0.80 * lgb_blend_rev 
        raw_cog = 0.10 * p_pr_cog + 0.10 * p_rd_cog + 0.80 * lgb_blend_cog

        self._submission = pd.DataFrame({
            "Date":    test_df["Date"].dt.strftime("%Y-%m-%d").values,
            "Revenue": _CAL_REV  * raw_rev,  
            "COGS":    _CAL_COGS * raw_cog,  
        })
        return self._submission

    def get_margin_fix(self) -> pd.DataFrame:
        sales = self._feat_df[["Date", "Revenue", "COGS"]].copy()
        sales["Q"] = sales["Date"].dt.quarter
        sales["Y"] = sales["Date"].dt.year
        recent_margin = {
            q: (sales[(sales["Q"] == q) & (sales["Y"] >= 2020)]["COGS"].sum() /
                sales[(sales["Q"] == q) & (sales["Y"] >= 2020)]["Revenue"].sum())
            for q in [1, 2, 3, 4]
        }
        result = self._submission.copy()
        target_cog_mean = result["COGS"].mean()
        result["_dt"] = pd.to_datetime(result["Date"])
        result["_q"]  = result["_dt"].dt.quarter
        for q in [1, 2, 3, 4]:
            mask = result["_q"] == q
            hist_cog = result.loc[mask, "Revenue"] * recent_margin[q]
            result.loc[mask, "COGS"] = (
                (1 - _BETA_MARGIN) * result.loc[mask, "COGS"] + _BETA_MARGIN * hist_cog
            )
        result["COGS"] *= (target_cog_mean / result["COGS"].mean())
        return result[["Date", "Revenue", "COGS"]]

    @staticmethod
    def _train_ridge(X, y, alpha=3.0):
        df = pd.DataFrame(X)
        mu, sigma = df.mean(axis=0), df.std(axis=0).replace(0, 1)
        model = Ridge(alpha=alpha, random_state=42)
        model.fit((df - mu) / sigma, y)
        return model, (mu, sigma)

    @staticmethod
    def _apply_ridge(model, stats, X):
        mu, sigma = stats
        df = pd.DataFrame(X)
        return model.predict((df - mu) / sigma)

    def _train_lgb(self, X, y, w, n_rounds=5000, early_stop=300):
        split_date = pd.Timestamp("2022-07-04")
        fit_mask = (self._feat_df["Date"] <= split_date).values
        val_mask = ~fit_mask
        probe = lgb.train(
            _LGB_PARAMS,
            lgb.Dataset(X[fit_mask], y[fit_mask], weight=w[fit_mask]),
            num_boost_round=n_rounds,
            valid_sets=[lgb.Dataset(X[val_mask], y[val_mask])],
            callbacks=[lgb.early_stopping(early_stop, verbose=False), lgb.log_evaluation(0)]
        )
        model = lgb.train(
            _LGB_PARAMS,
            lgb.Dataset(X, y, weight=w),
            num_boost_round=probe.best_iteration
        )
        return model, probe.best_iteration

    def _train_prophet(self, sales_df, target: str):
        log_target = np.log(sales_df[target])  
        train_df = pd.DataFrame({"ds": sales_df["Date"], "y": log_target}).merge(
            self.fe.build_promo_regressors(sales_df["Date"]), on="ds")
        train_df = train_df[train_df["ds"] >= "2020-01-01"]
        m = Prophet(
            yearly_seasonality=True, weekly_seasonality=True,
            daily_seasonality=False, seasonality_mode="multiplicative",
            changepoint_prior_scale=0.05
        )
        for col in [c for c in train_df.columns if c.startswith("promo_")]:
            m.add_regressor(col)
        m.fit(train_df)
        return m

    def _train_q_specialists(self):
        spec_rev, spec_cog = {}, {}
        Q_train = self._feat_df["Date"].dt.quarter.values
        for q in [1, 2, 3, 4]:
            w = self._sample_w.copy()
            w[Q_train == q] *= _Q_BOOST[q]
            spec_rev[q], _ = self._train_lgb(self._X_tr, self._y_rev, w, n_rounds=3000, early_stop=200)
            spec_cog[q], _ = self._train_lgb(self._X_tr, self._y_cog, w, n_rounds=3000, early_stop=200)
        return spec_rev, spec_cog

print('Forecasting Pipeline defined.')


Forecasting Pipeline defined.


In [33]:
# === 4. TRAIN & PREDICT ===
import os
from pathlib import Path

# Đường dẫn output tương đối (portable)
OUTPUT_DIR = Path.cwd().parent / 'output' / 'forecasting'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pipeline = ForecastingPipeline(fe)
pipeline.fit(sales)

test_dates = pd.date_range('2023-01-01', '2024-07-01', freq='D')
sub_base   = pipeline.predict(test_dates)
sub_margin = pipeline.get_margin_fix()

# Lưu file kết quả
base_path   = OUTPUT_DIR / 'submission.csv'
margin_path = OUTPUT_DIR / 'submission_margin.csv'

sub_base.to_csv(base_path, index=False)
sub_margin.to_csv(margin_path, index=False)

print(f"Revenue mean (base):   {sub_base['Revenue'].mean():,.0f}")
print(f"COGS mean    (base):   {sub_base['COGS'].mean():,.0f}")
print(f"Revenue mean (margin): {sub_margin['Revenue'].mean():,.0f}")
print(f"COGS mean    (margin): {sub_margin['COGS'].mean():,.0f}")
print(f'Saved: {base_path}')
print(f'Saved: {margin_path}')
sub_base.head(10)


Training Ridge...
Training LightGBM base...
Training Prophet...
Training Q-Specialists (8 models)...
All models trained.
Revenue mean (base):   4,140,280
COGS mean    (base):   3,815,846
Revenue mean (margin): 4,140,280
COGS mean    (margin): 3,815,846
Saved: /Users/tranquoctuan18112005/Documents/DATATHON/datathon-2026-round-1/bốn con cừu/output/forecasting/submission.csv
Saved: /Users/tranquoctuan18112005/Documents/DATATHON/datathon-2026-round-1/bốn con cừu/output/forecasting/submission_margin.csv


,Date,Revenue,COGS
0,2023-01-01,2.435848e+06,2.551448e+06
1,2023-01-02,1.774176e+06,1.830503e+06
2,2023-01-03,1.623380e+06,1.562856e+06
3,2023-01-04,1.194737e+06,1.100844e+06
4,2023-01-05,1.356783e+06,1.267918e+06
5,2023-01-06,1.697288e+06,1.549101e+06
6,2023-01-07,1.653254e+06,1.555883e+06
7,2023-01-08,1.781362e+06,1.756355e+06
8,2023-01-09,2.365607e+06,2.170657e+06
9,2023-01-10,2.123859e+06,1.949050e+06


In [34]:
# === 5. ĐÁNH GIÁ MÔ HÌNH: R², MAE, RMSE ===
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

val_data  = sales[sales['Date'].dt.year == 2022].copy()
pred_val  = pipeline.predict(val_data['Date'])

val_data  = val_data.set_index('Date')
pred_val  = pred_val.set_index('Date')

for col in ['Revenue', 'COGS']:
    y_true = val_data[col].values
    y_pred = pred_val[col].values
    r2   = r2_score(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f'{col:>10}  |  R²={r2:.4f}  |  MAE={mae:,.0f}  |  RMSE={rmse:,.0f}')


   Revenue  |  R²=0.6528  |  MAE=793,076  |  RMSE=986,298
      COGS  |  R²=0.5329  |  MAE=814,993  |  RMSE=996,837


In [35]:
# === 6. TRỰC QUAN HÓA XU HƯỚNG (Tách riêng 2 ảnh: Revenue & COGS) ===
import matplotlib.pyplot as plt
import pandas as pd
import os
from pathlib import Path

# Thư mục lưu ảnh
IMG_OUTPUT_DIR = Path.cwd().parent / 'output' / 'forecasting'
os.makedirs(IMG_OUTPUT_DIR, exist_ok=True)

# Chuẩn bị dữ liệu dự báo
sub_viz = sub_margin.copy()
sub_viz['Date'] = pd.to_datetime(sub_viz['Date'])

# --- ẢNH 1: DỰ BÁO DOANH THU (REVENUE) ---
plt.figure(figsize=(16, 6))
plt.plot(sales['Date'], sales['Revenue'], color='gray', alpha=0.5, label='Lịch sử', linewidth=1)
plt.plot(sub_viz['Date'], sub_viz['Revenue'], color='red', label='Dự báo 2023-2024', linewidth=1.5)

# Kẻ vạch năm
for year in range(sales['Date'].dt.year.min(), 2025):
    plt.axvline(pd.Timestamp(f'{year}-01-01'), color='black', linestyle='--', alpha=0.1)

plt.title('Dự báo Doanh thu (Revenue Trend 2023-2024)', fontsize=14, fontweight='bold', color='red')
plt.xlabel('Năm')
plt.ylabel('Giá trị (VND)')
plt.legend(loc='upper left')
plt.grid(alpha=0.2)
plt.savefig(IMG_OUTPUT_DIR / 'forecast_revenue_trend.png', bbox_inches='tight', dpi=300)
plt.show()

# --- ẢNH 2: DỰ BÁO GIÁ VỐN (COGS) ---
plt.figure(figsize=(16, 6))
plt.plot(sales['Date'], sales['COGS'], color='gray', alpha=0.5, label='Lịch sử', linewidth=1)
plt.plot(sub_viz['Date'], sub_viz['COGS'], color='green', label='Dự báo 2023-2024', linewidth=1.5)

# Kẻ vạch năm
for year in range(sales['Date'].dt.year.min(), 2025):
    plt.axvline(pd.Timestamp(f'{year}-01-01'), color='black', linestyle='--', alpha=0.1)

plt.title('Dự báo Giá vốn (COGS Trend 2023-2024)', fontsize=14, fontweight='bold', color='green')
plt.xlabel('Năm')
plt.ylabel('Giá trị (VND)')
plt.legend(loc='upper left')
plt.grid(alpha=0.2)
plt.savefig(IMG_OUTPUT_DIR / 'forecast_cogs_trend.png', bbox_inches='tight', dpi=300)
plt.show()

print(f"✅ Đã lưu 2 ảnh riêng biệt tại: {IMG_OUTPUT_DIR}")


✅ Đã lưu 2 ảnh riêng biệt tại: /Users/tranquoctuan18112005/Documents/DATATHON/datathon-2026-round-1/bốn con cừu/output/forecasting
